# Day 2




# Exploratory Data Analysis, Clustering, and Machine Learning Basics

**Goal:** visualize target and feature distributions, identify relationships, reduce
dimensionality with PCA, group complexes with K-Means, and build the vocabulary and
discipline needed before training a model on Day 3.

## Learning objectives

- Plot distributions and summary statistics of the target variable.
- Visualize pairwise relationships and compute correlation matrices (Pearson and Spearman).
- Use violin plots and 2D density plots to inspect conditional distributions.
- Save figures for use in reports and presentations.
- Apply Principal Component Analysis (PCA) to high-dimensional feature matrices.
- Interpret PCA loadings and visualize data in low-dimensional PCA space.
- Use the Elbow Method and Silhouette Score to select the number of clusters.
- Apply K-Means and characterize clusters by feature statistics.
- Distinguish supervised from unsupervised learning.
- Explain train/validation/test splits, data leakage, and why the test set is protected.

---

## Part 1 - Distribution analysis

### 1.1 Load the cleaned data and imports


### Set `REPO_ROOT`

Every notebook in this workshop locates the repository the same way: by walking up
from the notebook's own directory until it finds one containing both `data/` and
`notebooks/`.

Run this cell before any other code cell, and check that the printed path is your clone.


In [ ]:
# --- Set REPO_ROOT --------------------------------------------------------
# Locate the repository root by searching upward from this notebook's directory.
from pathlib import Path


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for d in (start, *start.parents):
        if (d / "data").is_dir() and (d / "notebooks").is_dir():
            return d
    raise FileNotFoundError(f"Repo root not found above {start}")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
FIG_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print("Repo root: ", REPO_ROOT)
print("Data dir:  ", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Figure dir:", FIG_DIR)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')

# CLEANED_PATH = Path(os.environ.get('SCRATCH', '.')) / 'Fe-Redox-GNN' / 'data_cleaned.csv'
CLEANED_PATH = OUTPUT_DIR / 'data_cleaned.pkl'
df = pd.read_pickle(CLEANED_PATH)
print('Loaded cleaned data with', len(df), 'rows from', CLEANED_PATH)
display(df.head())



### 1.2 Target distribution and moments

Plot the distribution of the redox target and compute skewness and kurtosis. The sample skewness and (excess) kurtosis are defined as:

$$
\text{skew}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{3}\right], \quad \text{kurt}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{4}\right] - 3
$$



#### What skewness and kurtosis look like

Two numbers on their own are hard to read, so here are four synthetic samples where the
answer is known in advance. All four are standardised to mean 0 and standard deviation 1,
so the centre and the width are identical in every panel and the only thing that differs is
shape. The dashed curve is the standard normal, the reference both moments are measured
against: by definition it has skew 0 and excess kurtosis 0.

- **Skewness** is about asymmetry. Zero means the two tails balance, positive means a longer
  right tail (a few unusually large values), negative means a longer left tail.
- **Excess kurtosis** is about tails more than peaks. Positive means heavier tails than a
  normal, more extreme values than you would expect, which also makes the peak sharper,
  since the standard deviation is fixed. Negative means the opposite: short tails and a flat
  top, as in the uniform case.

The two are not independent. Every distribution obeys $\mathrm{kurt} \ge \mathrm{skew}^2 - 2$,
so a strongly skewed sample is forced to carry some positive kurtosis too. That is why the
high-skew panel does not report a kurtosis near zero.

Compare all four against the two values printed for the real target above.


In [ ]:
# Mock distributions with known shapes, to calibrate what the two numbers above mean.
# Each sample is standardised to mean 0 and standard deviation 1, so any visible difference
# between the panels is skewness or kurtosis and nothing else. The seed is fixed so the
# figure is the same every run.

rng = np.random.default_rng(0)
N = 20000

samples = {
    'Low skew (symmetric)': rng.normal(0, 1, N),
    'High skew (long right tail)': rng.lognormal(0, 0.35, N),
    'Low kurtosis (flat, short tails)': rng.uniform(-1, 1, N),
    'High kurtosis (sharp peak, heavy tails)': rng.laplace(0, 1, N),
}

# the standard normal, drawn on every panel as the reference shape
x_ref = np.linspace(-4, 5, 400)
normal_pdf = np.exp(-x_ref ** 2 / 2) / np.sqrt(2 * np.pi)

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)

for ax, (label, raw) in zip(axes.ravel(), samples.items()):
    s = pd.Series((raw - raw.mean()) / raw.std())   # standardise: mean 0, std 1

    sns.histplot(s, bins=60, stat='density', kde=True, ax=ax)
    ax.plot(x_ref, normal_pdf, 'k--', lw=1.2, label='standard normal')

    # same .skew() and .kurt() used on the real target above
    ax.set_title(f'{label}\nskew = {s.skew():.2f}   excess kurtosis = {s.kurt():.2f}',
                 fontsize=11)
    ax.set_xlim(-4, 5)
    ax.set_xlabel('standardised value')
    ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()


### 1.3 Let's look at the redox potential distribution in our dataset

In [ ]:
TARGET_COL = 'redox_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f'{TARGET_COL} column not found in cleaned data')

plt.figure(figsize=(6,4))
sns.histplot(df[TARGET_COL], kde=True)
plt.xlim(-2,5)
plt.xlabel("Redox potential (V)")

plt.title('Redox potential distribution')
plt.savefig(FIG_DIR / 'redox_distribution.png', dpi=150)
plt.show()

print('skewness:', df[TARGET_COL].skew())
print('kurtosis (excess):', df[TARGET_COL].kurt())


### 1.4 Feature distributions

Visualize a small set of numeric features with histograms and KDEs. This helps spot multimodality and heavy tails.



In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['index', TARGET_COL]]

def plot_feature_distributions(df, cols):
    n = len(cols)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, c in zip(axes, cols):
        sns.histplot(df[c].dropna(), kde=True, ax=ax)
        ax.set_title(c)
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'feature_distributions.png', dpi=150)
    plt.show()

plot_feature_distributions(df, num_cols[:8])



---

## Part 2 - Relationship analysis

### 2.1 Scatter plots and Pearson correlation

Pearson correlation coefficient between two variables is defined as:

$$
r_{XY} = \frac{\sum_{i} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i} (x_i - \bar{x})^2}\ \sqrt{\sum_{i} (y_i - \bar{y})^2}}
$$

Plot a selection of features against the target and overlay a linear regression fit.



In [ ]:
pairs = [p for p in ['n_O', 'n_ligands', 'n_anionic'] if p in df.columns]
if not pairs:
    pairs = num_cols[:3]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 4), squeeze=False)
axes = np.atleast_1d(axes).flatten()
for ax, c in zip(axes, pairs):
    sns.regplot(x=c, y=TARGET_COL, data=df, ax=ax, scatter_kws={'s':10}, line_kws={'color':'red'})
    ax.set_title(f'{c} vs {TARGET_COL}')
for ax in axes[len(pairs):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'feature_vs_target_scatter.png', dpi=150)
plt.show()

for c in pairs:
    r = df[c].corr(df[TARGET_COL])
    print(f'Pearson r ({c} vs {TARGET_COL}):', r)



### 2.2 Pairplot for multi-feature relationships

A pairplot (scatter + KDE on diag) helps identify non-linear relationships and clusters.



In [ ]:
cols_for_pairplot = [c for c in ['n_O', 'n_ligands', 'n_anionic', 'num_atoms', 'q', TARGET_COL] if c in df.columns]
if len(cols_for_pairplot) > 1:
    sns.pairplot(df[cols_for_pairplot].dropna(), diag_kind='kde', corner=True)
    plt.savefig(FIG_DIR / 'pairplot.png', dpi=150)
    plt.show()



---

## Part 3 - Correlation analysis

### 3.1 Correlation matrix heatmap

Compute Pearson correlation matrix and plot a heatmap.



In [ ]:
corr = df[num_cols + [TARGET_COL]].corr()
plt.figure(figsize=(8,6),dpi=150)
sns.heatmap(corr, annot=True, fmt='.1f', cmap='vlag', center=0,
            annot_kws={'size': 6})
plt.title('Pearson correlation matrix')
plt.savefig(FIG_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()



### 3.2 Spearman rank correlation

Spearman rank correlation is useful for monotonic but non-linear relationships. The rank correlation coefficient is:

$$
\rho = 1 - \frac{6 \sum_{i} d_i^2}{n(n^2 - 1)}\quad\text{where } d_i = \mathrm{rank}(x_i) - \mathrm{rank}(y_i)
$$

Compute and compare Spearman to Pearson for the target relationships.



In [ ]:
spearman = df[num_cols + [TARGET_COL]].corr(method='spearman')
print(f'Top correlations with {TARGET_COL} (Pearson):')
print(corr[TARGET_COL].abs().sort_values(ascending=False).head(10))
print(f'\nTop correlations with {TARGET_COL} (Spearman):')
print(spearman[TARGET_COL].abs().sort_values(ascending=False).head(10))

# q and q_sum encode almost the same charge information. Keep one for modeling.
if 'q_sum' in num_cols:
    num_cols.remove('q_sum')
    print('\nDropping q_sum from model features because it duplicates q.')



---

## Part 4 - Advanced visualizations





### 4.1 2D density plot

A 2D kernel density estimate lets you visualize joint distributions and identify dense regions where models should focus their fit.



In [ ]:
if 'n_O' in df.columns and 'n_ligands' in df.columns:
    plt.figure(figsize=(6,5))
    sns.kdeplot(x=df['n_O'], y=df['n_ligands'], cmap='Blues', fill=True, thresh=0.05)
    plt.xlabel('n_O')
    plt.ylabel('n_ligands')
    plt.savefig(FIG_DIR / 'n_l_density.png', dpi=150)
    plt.show()


**Mentor checkpoint 4**: after the exploratory analysis (Parts 1-4)

After the exploratory analysis (Parts 1-4)

- Confirm whether the observed correlations align with chemical intuition.
- Discuss which features are plausible inputs for classical baselines (RF, GPR) vs. GNN inputs.
- Decide on a shortlist of features to use in Day 3 baselines.



## Part 5 - Theoretical background

### 5.1 The curse of dimensionality

When working with many features (e.g., dozens of structural descriptors for each iron complex) distances between points become less meaningful, models require exponentially more data to generalize, and visualization beyond 3D is impossible. Dimensionality reduction projects data from a high-dimensional space $\mathbb{R}^d$ to a lower-dimensional space $\mathbb{R}^k$ (where $k \ll d$) while preserving as much information as possible.

### 5.2 Principal Component Analysis (PCA)

PCA finds directions of maximum variance in the data. Given a centered data matrix $X \in \mathbb{R}^{n\times d}$, the covariance matrix is:

$$
C = \frac{1}{n-1} X^\top X
$$

PCA solves the eigenproblem

$$
C \mathbf{v}_i = \lambda_i \mathbf{v}_i, \qquad \lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_d
$$

and the principal component projection onto the top-$k$ eigenvectors $V_k = [\mathbf{v}_1,\dots,\mathbf{v}_k]$ is

$$
Z = X V_k
$$

The explained variance ratio for component $i$ is

$$
\mathrm{EVR}_i = \frac{\lambda_i}{\sum_{j=1}^d \lambda_j}
$$

### 5.3 K-Means clustering

K-Means partitions $n$ data points into $K$ clusters by minimizing the within-cluster sum of squares (WCSS):

$$
\mathrm{WCSS} = \sum_{k=1}^K \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2
$$

Algorithm (high level):

1. Initialize $K$ centroids (randomly or with k-means++).
2. Assign each point to the nearest centroid.
3. Update each centroid to be the mean of its assigned points.
4. Repeat steps 2–3 until convergence.

Use the Elbow Method (WCSS vs K) and the Silhouette Score to guide choice of $K$.

---



## Part 6 - Applying PCA

### 6.1 Data preparation

Load the cleaned dataset produced in Day 2 and select the engineered numeric features and target. `q_sum` is excluded because it is nearly identical to the overall charge `q`.



In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')

# WORKSHOP_PATH = os.path.join(os.environ.get('SCRATCH', '.'), 'Fe-Redox-GNN')
# df = pd.read_csv(os.path.join(WORKSHOP_PATH, 'data_cleaned.csv'))

CLEANED_PATH = OUTPUT_DIR / 'data_cleaned.pkl'
df = pd.read_pickle(CLEANED_PATH)

TARGET_COL = 'redox_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f"Target column '{TARGET_COL}' not found in data_cleaned.csv")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols

In [ ]:
df[numeric_cols]

In [ ]:
# df_clean = df.dropna()
# df_clean

In [ ]:

FEATURE_COLS = [
    c for c in numeric_cols
    if c not in {TARGET_COL, 'q_sum', 'n_C'} and df[c].nunique() > 1
]
if not FEATURE_COLS:
    raise RuntimeError('No non-constant numeric features found in data_cleaned.csv')

print(f"Dataset: {df.shape[0]} samples × {df.shape[1]} columns")
print(f"Target: {TARGET_COL}")
print(f"Number of features: {len(FEATURE_COLS)}")

display(df.head())



### 6.2 Standardize features (critical for PCA)

PCA is sensitive to feature scales; standardize to zero mean and unit variance using training statistics in modeling scenarios.



In [ ]:
scaler = StandardScaler()
X = df[FEATURE_COLS].values
X_scaled = scaler.fit_transform(X)

print('Before standardization (first 5 feature means):', np.round(X.mean(axis=0)[:5], 4))
print('After standardization (first 5 feature means):', np.round(X_scaled.mean(axis=0)[:5], 6))
print('After standardization (first 5 feature stds):', np.round(X_scaled.std(axis=0)[:5], 4))



### 6.3 Performing PCA and explained variance



In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Robust computation of components for thresholds
n_90 = (np.argmax(cumulative_var >= 0.90) + 1) if (cumulative_var >= 0.90).any() else len(explained_var)
n_95 = (np.argmax(cumulative_var >= 0.95) + 1) if (cumulative_var >= 0.95).any() else len(explained_var)

print('PCA Explained Variance (first 10 components):')
for i, (ev, cv) in enumerate(zip(explained_var[:10], cumulative_var[:10])):
    bar = '█' * int(ev * 50)
    marker = ' ◄── 90%' if i + 1 == n_90 else (' ◄── 95%' if i + 1 == n_95 else '')
    print(f'  PC{i+1:2d}: {ev:6.3f} ({cv:6.3f} cumulative) {bar}{marker}')

print(f'Components for 90% variance: {n_90}')
print(f'Components for 95% variance: {n_95}')

# Scree and cumulative plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(explained_var) + 1), explained_var, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot', fontweight='bold')

axes[1].plot(range(1, len(cumulative_var) + 1), cumulative_var, 'o-', color='steelblue', linewidth=2, markersize=6)
axes[1].axhline(y=0.90, color='red', linestyle='--', linewidth=1.5, label='90% variance')
axes[1].axhline(y=0.95, color='orange', linestyle='--', linewidth=1.5, label='95% variance')
axes[1].axvline(x=n_90, color='red', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_95, color='orange', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance', fontweight='bold')
axes[1].legend()

plt.suptitle('PCA Variance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()



### 6.4 Visualizing data in PCA space (2D)



In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

y = df[TARGET_COL].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
scatter1 = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='RdYlBu_r', alpha=0.6, s=30, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter1, ax=axes[0], label=TARGET_COL)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title(f'PCA Projection Colored by {TARGET_COL}', fontweight='bold')

sns.kdeplot(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1], fill=True, cmap='Blues', levels=15, ax=axes[1])
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
axes[1].set_title('PCA Density', fontweight='bold')

plt.suptitle('Iron Complexes in PCA Space', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pca_2d_projection.png', dpi=150, bbox_inches='tight')
plt.show()



### 6.5 Interpreting PCA loadings (biplot)



In [ ]:
loadings = pd.DataFrame(pca_2d.components_.T, columns=['PC1', 'PC2'], index=FEATURE_COLS)
print('PCA loadings (contribution of each feature to PCs):')
display(loadings)

# Biplot
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='RdYlBu_r', alpha=0.3, s=20)
scale = 3
for i, feature in enumerate(FEATURE_COLS):
    ax.annotate('', xy=(loadings.iloc[i, 0] * scale, loadings.iloc[i, 1] * scale), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.text(loadings.iloc[i, 0] * scale * 1.15, loadings.iloc[i, 1] * scale * 1.15, feature,
            fontsize=9, fontweight='bold', color='darkred', ha='center', va='center')

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA Biplot: Data Points + Feature Loadings', fontsize=13, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretation notes (discuss in mentor checkpoint):
# - Arrows pointing in the same direction indicate positively correlated features.
# - Opposite arrows indicate negative correlation.
# - Longer arrows indicate features with greater influence on the PCs.


**Reading the biplot**

Each arrow is one feature, drawn to its (PC1, PC2) loading. Direction shows which components the feature aligns with, length shows how well it is captured in this 2D view, and the angle between two arrows approximates their correlation: same direction means positively correlated, opposite means negatively correlated.

For this dataset:

- **PC1 is a size axis.** `num_atoms`, `n_H`, `fe_bond_mean` and `fe_bond_max` point together: larger complexes have more atoms, more hydrogens, and longer average Fe–ligand bonds.
- **PC2 is a charge axis.** `n_anionic` and `q` point in opposite directions: each anionic ligand added lowers the overall charge of the complex.
- The colour gradient runs diagonally: redox potential falls as complexes get larger and more anionic, consistent with anionic ligands stabilising Fe³⁺.

PC1 and PC2 together hold only ~38% of the variance, so arrow lengths and angles are a projection. A short arrow (e.g. `n_ligands`) means the feature varies mainly along components not shown here, not that it is unimportant.



---

## Part 7 - K-Means clustering

### 7.1 Elbow Method and Silhouette analysis

Silhouette coefficient for a sample $i$ is defined as

$$
s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}
$$

where $a(i)$ is the mean intra-cluster distance and $b(i)$ is the mean nearest-cluster distance.



In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f'K={k:2d}: WCSS={kmeans.inertia_:10.2f}, Silhouette={silhouettes[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Within-Cluster Sum of Squares (WCSS)')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xticks(list(K_range))
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontweight='bold')
axes[1].set_xticks(list(K_range))
axes[1].grid(True, alpha=0.3)

best_k = list(K_range)[int(np.argmax(silhouettes))]
axes[1].axvline(x=best_k, color='green', linestyle='--', linewidth=2, label=f'Best K = {best_k}')
axes[1].legend()

plt.suptitle('Optimal Number of Clusters', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Optimal K by Silhouette Score: {best_k}')


**Reading the elbow and silhouette plots**

WCSS is the sum of squared distances from each complex to its cluster centre. It always falls as K increases, so K is chosen at the "elbow", where the curve stops dropping steeply and extra clusters stop buying much.

The silhouette score compares each point's distance to its own cluster against its distance to the nearest other cluster, averaged over all points. Above ~0.5 means well-separated clusters; below ~0.25 means essentially none.

For this dataset:

- The WCSS curve has **no elbow**: successive drops decay smoothly, the signature of subdividing a continuous cloud rather than finding real groups.
- The silhouette score stays between 0.13 and 0.19 at every K, always below 0.25.
- **`best_k = 10` is therefore essentially meaningless.** The curve is flat, the differences between K = 6, 7 and 10 are in the third decimal, and the maximum sits at the edge of the search range, a sign that no real maximum was found. A different `random_state` or a wider `K_range` would return a different "optimal" K.

Iron complexes form a continuum in this feature space rather than discrete families. K-Means still returns a valid partition, but the groups are imposed, not discovered.



### 7.2 Clustering with optimal K and visualization



In [ ]:
optimal_k = 3 # since best_k = 10 is not something meaningful in out case
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
df['cluster'] = cluster_labels

# Visualize clusters in PCA space
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
colors = plt.cm.Set1(np.linspace(0, 1, optimal_k))
for k in range(optimal_k):
    mask = cluster_labels == k
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1], c=[colors[k]], label=f'Cluster {k} (n={mask.sum()})',
                    alpha=0.6, s=30, edgecolors='gray', linewidth=0.3)

centroids_pca = pca_2d.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black', marker='X', s=200, linewidths=2,
                edgecolors='white', zorder=5, label='Centroids')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
axes[0].set_title(f'K-Means Clusters (K={optimal_k}) in PCA Space', fontweight='bold')
axes[0].legend(fontsize=9, loc='best')

# Right: Redox potential distribution per cluster
sns.boxplot(data=df, x='cluster', y=TARGET_COL, palette='Set1', ax=axes[1])
sns.stripplot(data=df, x='cluster', y=TARGET_COL, color='black', alpha=0.3, size=3, jitter=True, ax=axes[1])
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel(TARGET_COL)
axes[1].set_title(f'{TARGET_COL} by Cluster', fontweight='bold')

plt.suptitle('Cluster Analysis of Iron Complexes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()



### 7.3 Cluster characterization



In [ ]:
cluster_stats = df.groupby('cluster')[FEATURE_COLS + [TARGET_COL]].agg(['mean', 'std'])
sizes = df.groupby('cluster').size()

print('Cluster Characterization:')
for k in range(optimal_k):
    print(f"\nCluster {k} (n = {sizes[k]} samples):")
    print(f"   {TARGET_COL}: {cluster_stats.loc[k, (TARGET_COL, 'mean')]:.4f} ± {cluster_stats.loc[k, (TARGET_COL, 'std')]:.4f}")
    for feat in FEATURE_COLS[:5]:
        print(f"   {feat}: {cluster_stats.loc[k, (feat, 'mean')]:.4f} ± {cluster_stats.loc[k, (feat, 'std')]:.4f}")

centroid_df = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=FEATURE_COLS, index=[f'Cluster {k}' for k in range(optimal_k)])
fig, ax = plt.subplots(figsize=(12, max(3, optimal_k)))
sns.heatmap(centroid_df.T, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Cluster Centroids (Original Feature Scale)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'cluster_centroids_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


---

## Part 8 - Machine learning basics

Everything so far has been description: what the data looks like, how features relate to each
other, which complexes group together. Day 3 switches to prediction. This part introduces the
vocabulary and, more importantly, the discipline that separates a believable result from a
flattering one.

### 8.1 Supervised and unsupervised learning

**Supervised learning** means every training example carries a known answer, and the model
learns a mapping from inputs to that answer. It splits by what the answer looks like:

- **Regression** predicts a continuous number. Predicting `redox_pot` in volts from the
  structural features of a complex is a regression problem, and it is what Day 3 does with
  Random Forest and Gaussian Process Regression.
- **Classification** predicts a category. "Will this complex have a potential above 1 V?" or
  "which ligand class dominates this complex?" are classification problems. The output is a
  label, and accuracy is measured by how often the label is right rather than by how close a
  number is.

**Unsupervised learning** has no answer key. The algorithm is given the inputs alone and asked
to find structure. You have already done two kinds of it in this notebook:

- **Dimensionality reduction**: PCA in Part 6. Nothing told PCA about the redox potential; it
  only found the directions along which the features vary most.
- **Clustering**: K-Means in Part 7. Again the target was never used. The clusters came out of
  feature similarity, and the fact that they *also* separate in redox potential is a finding,
  not something the algorithm was aiming at.
- **Neighbour methods** sit on the boundary and are worth pinning down, because the name is
  used for both. Nearest-neighbour *search*, "which complexes most resemble this one?", is
  unsupervised, and is how similarity searches over fingerprints work. k-Nearest-Neighbours
  *prediction*, where you average the known targets of the k closest points, is supervised,
  because it uses the answers.

The distinction matters for evaluation. A supervised model can be scored against held-out
truth. An unsupervised result has no truth to check against, which is why Part 7 needed
proxies like the silhouette score and, ultimately, chemical judgement.

### 8.2 Train, validation, and test

A model's error on the data it was fitted to is not an estimate of anything useful. Any model
with enough capacity can memorise the training set, and a memorised answer says nothing about
a complex it has never seen. So the data is divided:

| Split | Used for | Seen by the model? |
|---|---|---|
| **Training** | fitting parameters | yes, directly |
| **Validation** | choosing hyperparameters, comparing candidate models, deciding when to stop | indirectly, through your decisions |
| **Test** | one final estimate of performance on unseen data | never, until the very end |

Training and test are the minimum. Validation is what keeps the test set clean: every time you
compare two models and keep the better one, you are fitting your *choices* to whatever data
produced that comparison. Do that on the test set and the test score stops being an estimate
of future performance and starts being a best case.

Cross-validation is the usual refinement. Rather than sacrificing a fixed slice to validation,
k-fold CV rotates the validation slice through the training data and averages the result, which is more
reliable on a dataset this size, where a single 20% split is small enough to be noisy.

### 8.3 Data leakage

**Leakage is any path by which information that would not be available at prediction time
reaches the model.** It inflates your scores, and it is dangerous precisely because it looks
like success: nothing errors, and the numbers improve.

The common routes:

- **Preprocessing fitted on everything.** Calling `scaler.fit_transform(X)` on the whole matrix
  before splitting lets the training set see the test set's mean and variance. The same applies
  to imputing missing values, PCA, and any other fitted transform. The correct pattern is
  `fit` on train, `transform` on test, which is exactly what the Day 3 setup does.
- **Feature selection on the full dataset.** Ranking features by their correlation with the
  target over all rows, then splitting, selects features using answers the model should not
  have had. The selection has to happen inside the training fold.
- **Duplicates spanning the split.** If the same complex, or a near-identical one, lands in both
  train and test, the test score measures recall rather than generalisation. This is a real risk
  for a chemical dataset built from a structural database, where the same ligand set recurs.
- **Target leakage.** A feature computed from the target, or from something only known after the
  target is known, will predict it beautifully and be useless in production.
- **Tuning against the test set.** Running the test set repeatedly and adjusting after each
  look is leakage through you rather than through the code, and no library can catch it.

Which is why the test set is treated as write-once. You get one honest look. Once you have seen
it and changed anything as a result, it has quietly become a validation set, and there is no
procedure that undoes this; the only remedy is data you have not touched.

One honest caveat about this notebook: the EDA in Parts 1-4 examined the full dataset, so your
own understanding of the data is already informed by rows that will end up in the test set.
That is normal practice and usually tolerated, but it is the same mechanism in a weaker form,
and it is worth knowing that the line is a matter of degree rather than a hard boundary.

### 8.4 Why the final model is trained on everything

There is an apparent contradiction at the end of a project. You carefully protect a test set,
measure your model on it once, and then, to deploy, you retrain on all the data including that
test set. That is correct, and the reason is what the test set was actually measuring.

The held-out evaluation estimates the performance of a **procedure**: this model class, these
hyperparameters, this preprocessing, trained on roughly this much data, not of one particular
fitted object. Once that estimate exists and every choice is frozen, refitting the identical
procedure on 100% of the data instead of 75% gives a model that is, on average, slightly
better, because more training data is the one reliable improvement available. The number you
report is still the held-out one; it describes what you should expect from the procedure.

Two conditions make this safe, and both are easy to violate:

1. **Every decision must already be frozen.** Model class, hyperparameters, feature set,
   preprocessing. If you retrain on everything and then keep tuning, you have no clean estimate
   left, because there is nothing held out to produce one.
2. **The procedure must stay identical.** Refit the scaler on the full data, keep the same
   hyperparameters. Changing anything at this step invalidates the estimate you just measured.

The trade-off you accept is that the production model can no longer be evaluated honestly:
there is no unseen data left. That is the price of using all of it, and it is why the retrain
happens last, once, and never as part of the experimentation loop.


---


**Mentor checkpoint 5**: after PCA, clustering, and ML basics (Parts 5-8)

After PCA, clustering, and ML basics (Parts 5-8)

- Confirm how many principal components are needed to capture 90% of the variance for your dataset.
- Discuss which physical features dominate PC1 and PC2 and whether they align with chemical intuition.
- Do the clusters correspond to chemically meaningful groups (ligand types, charge states, etc.)?
- Is there a clear separation in redox potential between clusters?
- State, in your own words, one way leakage could occur in this dataset and how you would prevent it.

Proceed only after confirmation.


---

## Exercises

### Exercise 1 - Custom correlation heatmap

Create a custom heatmap focusing on the top 10 features most correlated (absolute Pearson r) with `redox_pot`. Save it as `FIG_DIR / 'exercise_1_heatmap.png'`.

Provide code that computes the top features, recomputes the correlation submatrix, and plots it with annotations.

### Exercise 2 - Feature relationship deep dive

Pick one feature (for example `fe_bond_mean`) and explore polynomial relationships to the redox target. Fit polynomial models of degree 1..5 and report validation R² for each.

Coefficient of determination (R²):

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

Hint: Split the data into a train/validation split (e.g., 80/20) or use k-fold cross-validation to compare polynomial degrees fairly and avoid overfitting.

### Exercise 3 - Discussion

Answer in Markdown: based on the EDA, what modeling approach would you try first and why? Consider model complexity, interpretability, and dataset size.

### Exercise 4 - Elbow Method (Required)

1. Run K-Means for K = 2 to 12.
2. Plot both the Elbow curve and Silhouette scores.
3. Justify your choice of optimal K in a Markdown answer.

### Exercise 5 - 3D PCA Visualization

1. Fit PCA with 3 components.
2. Create an interactive 3D scatter plot using matplotlib's Axes3D and color points by cluster assignment.
3. Report how much additional variance PC3 captures.

### Exercise 6 - t-SNE Comparison (Bonus)

t-SNE optimizes a KL divergence between high- and low-dimensional pairwise distributions:

$$
\mathrm{KL}(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}
$$

1. Apply t-SNE with perplexity=30 and compare the 2D embedding to PCA for cluster separation.
2. Discuss why t-SNE may be better for visualization but worse as a preprocessing step for ML.


---

## Summary

**Visualization**

| Visualization | Purpose | Library |
|---|---:|---|
| Histogram + KDE | Inspect target distribution, skewness | seaborn |
| Scatter + regplot | Inspect linear trends with target | seaborn |
| Pairplot | Multi-feature relationships | seaborn |
| Correlation heatmap | Global linear associations | seaborn, matplotlib |
| Violin / KDE / 2D density | Conditional distributions and joint density | seaborn |

**Dimensionality reduction and clustering**

| Method | Purpose | Key parameter |
|---|---:|---|
| StandardScaler | Normalize features to zero mean, unit variance | - |
| PCA | Linear dimensionality reduction | n_components |
| K-Means | Partition data into K clusters | n_clusters |
| Elbow Method | Find K by WCSS vs K | WCSS vs K |
| Silhouette Score | Evaluate cluster quality | Range: [-1, 1] |

**Machine learning vocabulary**

| Term | Meaning |
|---|---|
| Supervised | Learns from labelled examples; regression predicts a number, classification a category |
| Unsupervised | No labels; PCA and K-Means above are both examples |
| Training set | Data the model is fitted on |
| Validation set | Data used to choose hyperparameters and compare models |
| Test set | Held back for a single, final, honest estimate |
| Data leakage | Information reaching the model that would not exist at prediction time |
| Final retrain | After all choices are frozen, refit the same procedure on all data |

---
